# Clustering, Under the Hood
### A 5-minute deep dive: K-Means and DBSCAN, step by step

**Agenda**
1. The data (and why it's designed to break naive assumptions)
2. K-Means, implemented from scratch — watch it iterate
3. Why initialization matters (and what K-Means++ fixes)
4. Where K-Means fundamentally fails (non-convex shapes)
5. DBSCAN, implemented from scratch — density instead of distance-to-centroid
6. Picking the number of clusters: inertia, elbow, silhouette

> Run all cells top to bottom before presenting. During the talk, jump straight to the plots — the markdown is here so you (or the audience) can read the *why* at your own pace.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

rng = np.random.default_rng(42)
plt.rcParams["figure.dpi"] = 110


## 1. The data

Two datasets, on purpose:

- **`blobs`** — convex, roughly spherical, well-separated. This is the "easy mode" every clustering demo uses, and the *only* regime where K-Means' assumptions actually hold.
- **`moons`** — two interleaved crescents. No linear boundary separates them, and neither cluster is convex. This is where distance-to-a-single-centroid breaks down, and it's the whole reason density-based methods exist.

Keeping both on screen at once is the point: the audience should see *one algorithm doesn't dominate the other* — the right tool depends on the geometry of the data.


In [ ]:
X_blobs, y_blobs_true = make_blobs(n_samples=300, centers=4, cluster_std=0.70, random_state=42)
X_moons, y_moons_true = make_moons(n_samples=300, noise=0.06, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], s=18, c="steelblue")
axes[0].set_title("blobs — convex, separable")
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], s=18, c="darkorange")
axes[1].set_title("moons — non-convex, interleaved")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()


## 2. K-Means from scratch

K-Means minimizes one objective, the **within-cluster sum of squares (inertia)**:

$$J = \sum_{i=1}^{n} \big\| x_i - \mu_{c(i)} \big\|^2$$

where $\mu_{c(i)}$ is the centroid of the cluster point $x_i$ is assigned to. It's a chicken-and-egg problem — the best assignment depends on the centroids, and the best centroids depend on the assignment — so K-Means solves it by **coordinate descent**, alternating two steps until nothing moves:

1. **Assignment step** — freeze the centroids, assign every point to its nearest one:
   $$c(i) = \arg\min_k \| x_i - \mu_k \|^2$$
2. **Update step** — freeze the assignment, move each centroid to the mean of its points:
   $$\mu_k = \frac{1}{|S_k|} \sum_{i \in S_k} x_i$$

Each of these steps can only decrease or hold $J$ steady — never increase it — which is why K-Means always converges. It just isn't guaranteed to reach the *global* minimum (more on that in section 3).

Below is a bare-bones implementation that records its full history so we can watch it converge.


In [ ]:
def kmeans_from_scratch(X, k, init_centroids, max_iter=20, tol=1e-6):
    """Lloyd's algorithm. Returns the full history of (centroids, labels, inertia)
    so every iteration can be inspected/plotted afterward."""
    centroids = init_centroids.copy()
    history = []

    for it in range(max_iter):
        # --- assignment step: distance from every point to every centroid ---
        dists = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)  # (n, k)
        labels = dists.argmin(axis=1)
        inertia = ((X - centroids[labels]) ** 2).sum()
        history.append((centroids.copy(), labels.copy(), inertia))

        # --- update step: move each centroid to the mean of its assigned points ---
        new_centroids = np.array([
            X[labels == j].mean(axis=0) if np.any(labels == j) else centroids[j]
            for j in range(k)
        ])

        if np.linalg.norm(new_centroids - centroids) < tol:
            centroids = new_centroids
            break
        centroids = new_centroids

    return history

k = 4
init_idx = rng.choice(len(X_blobs), size=k, replace=False)
init_centroids = X_blobs[init_idx]

history = kmeans_from_scratch(X_blobs, k, init_centroids)
print(f"Converged in {len(history)} iterations, final inertia = {history[-1][2]:.1f}")


### Watching it converge

Every panel below is one full assignment+update step. Stars are centroids. Notice how the boundaries snap into place within the first 2-3 iterations — that's typical, K-Means converges fast on well-separated data.


In [ ]:
n_show = min(6, len(history))
fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 3.2))
if n_show == 1:
    axes = [axes]

for i, (ax, (centroids, labels, inertia)) in enumerate(zip(axes, history[:n_show])):
    ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap="tab10", s=14, alpha=0.8)
    ax.scatter(centroids[:, 0], centroids[:, 1], c="black", marker="*", s=180, edgecolor="white", linewidth=1)
    ax.set_title(f"iter {i}  (J={inertia:.0f})")
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle("K-Means: assignment + update, iteration by iteration")
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 3.2))
plt.plot([h[2] for h in history], marker="o")
plt.xlabel("iteration")
plt.ylabel("inertia J (within-cluster SSE)")
plt.title("The objective can only go down")
plt.tight_layout()
plt.show()


## 3. Initialization matters: why "K-Means" usually means "K-Means++"

Lloyd's algorithm is a **local** optimizer — coordinate descent only guarantees it won't get worse, not that it finds the global minimum of $J$. Bad luck in the initial centroids can lock it into a poor local optimum, especially two centroids landing in the same true cluster.

**K-Means++** fixes this by making initialization *probability-weighted by distance*: the first centroid is picked uniformly at random, then each subsequent centroid is sampled with probability proportional to $D(x)^2$ — the squared distance to the nearest centroid already chosen. Points far from existing centroids are far more likely to be picked, which spreads the initial centroids out and makes bad local optima much rarer (this is also what `sklearn`'s `KMeans` uses by default).

Below: same data, same algorithm, only the initialization differs.


In [ ]:
def kmeans_pp_init(X, k, rng):
    centroids = [X[rng.integers(len(X))]]
    for _ in range(1, k):
        d2 = np.min([np.sum((X - c) ** 2, axis=1) for c in centroids], axis=0)
        probs = d2 / d2.sum()
        next_idx = rng.choice(len(X), p=probs)
        centroids.append(X[next_idx])
    return np.array(centroids)

# A deliberately bad random init: all 4 centroids sampled from a tight corner
bad_rng = np.random.default_rng(7)
corner_mask = (X_blobs[:, 0] < X_blobs[:, 0].mean()) & (X_blobs[:, 1] < X_blobs[:, 1].mean())
bad_init = X_blobs[corner_mask][bad_rng.choice(corner_mask.sum(), size=k, replace=False)]

good_init = kmeans_pp_init(X_blobs, k, np.random.default_rng(7))

hist_bad = kmeans_from_scratch(X_blobs, k, bad_init)
hist_good = kmeans_from_scratch(X_blobs, k, good_init)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, hist, title in zip(axes, [hist_bad, hist_good], ["random init (unlucky)", "K-Means++ init"]):
    centroids, labels, inertia = hist[-1]
    ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap="tab10", s=14, alpha=0.8)
    ax.scatter(centroids[:, 0], centroids[:, 1], c="black", marker="*", s=180, edgecolor="white")
    ax.set_title(f"{title}\nfinal J={inertia:.0f}")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()


## 4. Where K-Means fundamentally fails

This isn't a tuning problem — it's structural. K-Means assigns every point to its nearest **centroid**, which means its decision boundaries are always a Voronoi diagram: straight lines (hyperplanes). It has no way to represent a boundary that curves around another cluster, no matter how it's initialized or how many iterations it runs.

Watch what happens on `moons`.


In [ ]:
k_moons = 2
init_moons = kmeans_pp_init(X_moons, k_moons, np.random.default_rng(0))
hist_moons = kmeans_from_scratch(X_moons, k_moons, init_moons)
centroids_m, labels_m, inertia_m = hist_moons[-1]

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons_true, cmap="coolwarm", s=16)
axes[0].set_title("ground truth")
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_m, cmap="coolwarm", s=16)
axes[1].scatter(centroids_m[:, 0], centroids_m[:, 1], c="black", marker="*", s=180, edgecolor="white")
axes[1].set_title(f"K-Means result (J={inertia_m:.1f})\nstraight-line boundary, wrong clusters")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()


## 5. DBSCAN from scratch: density instead of distance-to-centroid

DBSCAN never computes a centroid. Instead it grows clusters through **density-reachability**, using two parameters:

- **`eps`** — the neighborhood radius.
- **`min_samples`** — how many neighbors (including itself) a point needs within `eps` to count as **dense**.

Every point falls into one of three roles:

- **Core point** — has at least `min_samples` points within `eps` of it.
- **Border point** — not a core point itself, but lies within `eps` of a core point.
- **Noise** — neither. Unlike K-Means, DBSCAN can simply refuse to assign a point to any cluster.

The algorithm: pick an unvisited core point, start a new cluster, then **flood-fill** — recursively pull in every point density-reachable from it (any point within `eps` of a core point already in the cluster). When a core point's neighbors are exhausted, move to the next unvisited core point and start a new cluster. This region-growing is exactly why DBSCAN can trace an arbitrarily curved shape: it never assumes clusters are convex, it just follows density wherever it leads.


In [ ]:
def dbscan_from_scratch(X, eps, min_samples):
    n = len(X)
    labels = np.full(n, -1)          # -1 = noise (default, may be reclassified as border)
    visited = np.zeros(n, dtype=bool)
    nbrs = NearestNeighbors(radius=eps).fit(X)
    neighbors = nbrs.radius_neighbors(X, return_distance=False)

    cluster_id = -1
    for i in range(n):
        if visited[i]:
            continue
        visited[i] = True
        if len(neighbors[i]) < min_samples:
            continue  # not a core point (yet) -> stays noise unless reached later as a border point

        cluster_id += 1
        seeds = list(neighbors[i])
        labels[i] = cluster_id
        j = 0
        while j < len(seeds):
            p = seeds[j]
            if not visited[p]:
                visited[p] = True
                if len(neighbors[p]) >= min_samples:
                    seeds.extend(neighbors[p])  # p is core too -> expand the flood-fill
            if labels[p] == -1:
                labels[p] = cluster_id          # reached as a border (or newly-found core) point
            j += 1

    return labels

eps, min_samples = 0.20, 5
labels_dbscan = dbscan_from_scratch(X_moons, eps, min_samples)
n_clusters = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
n_noise = (labels_dbscan == -1).sum()
print(f"DBSCAN found {n_clusters} clusters and {n_noise} noise points")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_m, cmap="coolwarm", s=16)
axes[0].set_title("K-Means (straight-line boundary)")

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_dbscan, cmap="coolwarm", s=16)
axes[1].set_title(f"DBSCAN (eps={eps}, min_samples={min_samples})\nfollows density, curves included")

noise_mask = labels_dbscan == -1
axes[2].scatter(X_moons[~noise_mask, 0], X_moons[~noise_mask, 1], c=labels_dbscan[~noise_mask], cmap="coolwarm", s=16)
axes[2].scatter(X_moons[noise_mask, 0], X_moons[noise_mask, 1], c="lightgray", s=30, marker="x", label="noise")
axes[2].set_title("noise points highlighted")
axes[2].legend(loc="lower right", fontsize=8)

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()


## 6. Picking the number of clusters

DBSCAN sidesteps "choose k" entirely — clusters emerge from density. K-Means doesn't have that luxury; `k` is a required input. Two classic diagnostics, both computable from what we already built:

**Elbow method** — plot inertia $J$ vs. $k$. $J$ always decreases as $k$ grows (more centroids can only fit the data better), but the *rate* of decrease drops sharply once $k$ passes the true cluster count — that bend is the "elbow."

**Silhouette score** — for each point, compare its average distance to points in its own cluster ($a$) against its average distance to points in the *nearest other* cluster ($b$):
$$s = \frac{b - a}{\max(a, b)} \in [-1, 1]$$
Near $+1$ means well-clustered, near $0$ means on a boundary, negative means likely misassigned. Unlike inertia, this doesn't automatically favor larger $k$, so it's a sharper signal for the right cluster count.


In [ ]:
ks = range(2, 9)
inertias, silhouettes = [], []

for kk in ks:
    init = kmeans_pp_init(X_blobs, kk, np.random.default_rng(1))
    hist = kmeans_from_scratch(X_blobs, kk, init)
    _, labels_k, inertia_k = hist[-1]
    inertias.append(inertia_k)
    silhouettes.append(silhouette_score(X_blobs, labels_k))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].plot(list(ks), inertias, marker="o")
axes[0].set_xlabel("k"); axes[0].set_ylabel("inertia"); axes[0].set_title("Elbow method")

axes[1].plot(list(ks), silhouettes, marker="o", color="darkorange")
axes[1].set_xlabel("k"); axes[1].set_ylabel("mean silhouette"); axes[1].set_title("Silhouette score")
best_k = list(ks)[int(np.argmax(silhouettes))]
axes[1].axvline(best_k, color="gray", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()
print(f"Silhouette picks k = {best_k} (true k = 4)")


## Takeaways

- K-Means is **coordinate descent on inertia** — fast, simple, but only correct when clusters are convex and roughly equal-spread, and it needs `k` in advance.
- Initialization isn't a footnote: **K-Means++**'s distance-weighted sampling is what makes the algorithm reliable in practice.
- DBSCAN trades "distance to a centroid" for **density-reachability**, which is why it handles arbitrary shapes and gets noise-rejection for free — at the cost of two new hyperparameters (`eps`, `min_samples`) that are themselves not trivial to choose.
- No free lunch: pick the algorithm based on the geometry you expect, and use inertia/elbow/silhouette as *evidence*, not as an oracle.
